In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

In [0]:
bronze_df = spark.read.table(f"adb_caresync_smd.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")

In [0]:
from pyspark.sql import functions as  F

updated_df = bronze_df.withColumn("insert_timestamp", F.current_timestamp()).withColumn("update_timestamp", F.current_timestamp())

In [0]:
from delta.tables import DeltaTable

target_table = f"adb_caresync_smd.silver.{table_name}"
source_record_count = updated_df.count()

print(f"={'=' * 60}")
print(f"  BRONZE -> SILVER LOAD")
print(f"={'=' * 60}")
print(f"  Table Name     : {table_name}")
print(f"  Target Table   : {target_table}")
print(f"  Load Type      : {load_type}")
print(f"  Merge Keys     : {merge_keys if merge_keys else 'N/A'}")
print(f"  Source Records : {source_record_count}")
print(f"  Source Columns : {', '.join(updated_df.columns)}")
print(f"={'=' * 60}")

if load_type == "FULL":
    print(f"\n[FULL LOAD] Overwriting target table '{target_table}' with {source_record_count} records...")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print(f"[FULL LOAD] Successfully overwrote '{target_table}'.")

elif load_type == "APPEND":
    print(f"\n[APPEND] Appending {source_record_count} records to '{target_table}'...")
    updated_df.write.mode("append").saveAsTable(target_table)
    print(f"[APPEND] Successfully appended {source_record_count} records to '{target_table}'.")

elif load_type == "MERGE":
    print(f"\n[MERGE] Checking if target table '{target_table}' exists...")

    if spark.catalog.tableExists(target_table):
        print(f"[MERGE] Target table exists. Proceeding with MERGE (upsert).")
        delta_table = DeltaTable.forName(spark, target_table)

        target_record_count = delta_table.toDF().count()
        print(f"[MERGE] Existing target record count: {target_record_count}")

        keys = [k.strip() for k in merge_keys.split(",")]
        merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in keys])
        print(f"[MERGE] Merge keys: {keys}")
        print(f"[MERGE] Merge condition: {merge_condition}")

        # On update: refresh all columns except insert_timestamp; set update_timestamp to now
        update_columns = [c for c in updated_df.columns if c != "insert_timestamp"]
        update_set = {c: F.col(f"source.{c}") for c in update_columns}
        update_set["update_timestamp"] = F.current_timestamp()

        print(f"[MERGE] Columns to update on MATCH: {update_columns}")
        print(f"[MERGE] 'insert_timestamp' will be preserved for matched (existing) records.")
        print(f"[MERGE] 'update_timestamp' will be set to current_timestamp() for matched records.")
        print(f"[MERGE] All columns will be inserted for NOT MATCHED (new) records.")

        print(f"\n[MERGE] Executing merge...")
        delta_table.alias("target").merge(
            updated_df.alias("source"),
            merge_condition
        ).whenMatchedUpdate(set=update_set) \
         .whenNotMatchedInsertAll() \
         .execute()

        final_count = spark.read.table(target_table).count()
        print(f"[MERGE] Merge completed successfully.")
        print(f"[MERGE] Final target record count: {final_count}")
        print(f"[MERGE] Net new records added: {final_count - target_record_count}")
    else:
        print(f"[MERGE] Target table does NOT exist. Creating it with initial {source_record_count} records...")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"[MERGE] Created '{target_table}' with {source_record_count} records.")

else:
    raise ValueError(f"Unsupported load_type: '{load_type}'. Expected 'FULL', 'APPEND', or 'MERGE'.")

print(f"\n{'=' * 61}")
print(f"  LOAD COMPLETE")
print(f"{'=' * 61}")

In [0]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))